In [1]:
import pandas as pd

df = pd.read_csv("cleaned_dataset_sample.csv")

# Create target column
df['High_Volume'] = (df['volume'] > df['volume'].median()).astype(int)

print(df[['volume', 'High_Volume']].head())

    volume  High_Volume
0  1123277            0
1  1955639            0
2  1645312            0
3   800809            0
4  2001050            0


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [3]:
X = df[['open', 'high', 'low', 'close']]
y = df['High_Volume']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
lr = LogisticRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [7]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

In [8]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [9]:
def evaluate_model(y_test, y_pred, model_name):
    print(f"\n{model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))

evaluate_model(y_test, lr_pred, "Logistic Regression")
evaluate_model(y_test, dt_pred, "Decision Tree")
evaluate_model(y_test, rf_pred, "Random Forest")


Logistic Regression
Accuracy: 0.619
Precision: 0.6023294509151415
Recall: 0.7182539682539683
F1 Score: 0.6552036199095023

Decision Tree
Accuracy: 0.569
Precision: 0.5762004175365344
Recall: 0.5476190476190477
F1 Score: 0.5615462868769074

Random Forest
Accuracy: 0.625
Precision: 0.6319018404907976
Recall: 0.6130952380952381
F1 Score: 0.622356495468278


In [10]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1'
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

Best Parameters: {'max_depth': 10, 'n_estimators': 100}


In [11]:
best_rf = grid.best_estimator_
best_pred = best_rf.predict(X_test)

evaluate_model(y_test, best_pred, "Optimized Random Forest")


Optimized Random Forest
Accuracy: 0.635
Precision: 0.6547884187082406
Recall: 0.5833333333333334
F1 Score: 0.6169989506820567
